In [1]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
import re
import unicodedata
import html

/home/onyxia/work/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. HTML entities (&gt etc.)
    text = html.unescape(text)

    # 2. Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # 3. Fix escaped apostrophes (IMPORTANT)
    text = text.replace("\\'", "'")

    # 4. Remove leftover backslashes
    text = text.replace("\\", "")

    # 5. Fix non-breaking spaces
    text = text.replace("\xa0", " ")

    # 6. Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [9]:
v_clean = np.vectorize(clean_text)

### Import both models

In [3]:
classifier = XGBClassifier()
classifier.load_model("model.json")

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

sbert = SentenceTransformer(MODEL_NAME,
    device=device)

cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4922.31it/s]


### Import data to infer

In [5]:
df = pd.read_csv('flat_french_political_interactions.csv')
print(len(df))
all_interactions = df['text'].tolist()

109133


In [6]:
N_CHUNKS = 20
parts = np.array_split(all_interactions, N_CHUNKS)
print(f"Chunk sizes: {[len(p) for p in parts]}")

Chunk sizes: [5457, 5457, 5457, 5457, 5457, 5457, 5457, 5457, 5457, 5457, 5457, 5457, 5457, 5456, 5456, 5456, 5456, 5456, 5456, 5456]


### Infer

In [7]:
def predict_with_threshold(model, X, threshold=0.5):

    proba = model.predict_proba(X)[:, 1]

    return (proba >= threshold).astype(int)

In [10]:
best_treshold = 0.5289310689146067
for i in range(1, N_CHUNKS + 1):
    chunk_text = parts[i - 1].copy()
    chunk_clean = v_clean(chunk_text)
    X = sbert.encode(
        chunk_clean,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    y_pred = predict_with_threshold(classifier, X, best_treshold)
    y_labels = ['Left' if l==0 else 'Right' for l in y_pred]

    temp_df = pd.DataFrame(chunk_text, columns=['text'])
    temp_df['left_right'] = y_labels
    temp_df.to_csv(f'data_left_right_{i}.csv', index=False)
    print(temp_df['left_right'].value_counts()/len(temp_df))

Batches: 100%|██████████| 22/22 [00:44<00:00,  2.04s/it]


left_right
Left     0.612424
Right    0.387576
Name: count, dtype: float64


Batches: 100%|██████████| 22/22 [00:38<00:00,  1.74s/it]


left_right
Left     0.597764
Right    0.402236
Name: count, dtype: float64


Batches: 100%|██████████| 22/22 [00:39<00:00,  1.81s/it]


...

left_right
Left     0.57423
Right    0.42577
Name: count, dtype: float64


Batches: 100%|██████████| 22/22 [00:38<00:00,  1.75s/it]


left_right
Left     0.642045
Right    0.357955
Name: count, dtype: float64


Batches: 100%|██████████| 22/22 [00:37<00:00,  1.69s/it]


left_right
Left     0.613453
Right    0.386547
Name: count, dtype: float64
